In [ ]:
为实现ckpt,训练脚本中,你需要做两件事:

1.开局检测：看看有没有之前的存档，有就读档。
2.定时存档：每练完一轮，就把模型、优化器、进度都打包存起来。

In [5]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib
import shap

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"当前使用的设备: {device}")

# ==========================================
# 1. 准备原始数据
# ==========================================
X, y = shap.datasets.adult()
# 只取数值特征方便演示
numeric_cols = ['Age', 'Education-Num', 'Capital Gain', 'Capital Loss', 'Hours per week']
X_data = X[numeric_cols].values
y_data = y.astype(float)

# 标准化 (神经网络必须做)
scaler = StandardScaler()
X_data = scaler.fit_transform(X_data)

X_train, X_test, y_train, y_test = train_test_split(X_data, y_data, test_size=0.2)

# ==========================================
# 2. 定义 Dataset (仓库：告诉程序怎么取一条数据)
# ==========================================
class MyIncomeDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.FloatTensor(features)
        self.labels = torch.FloatTensor(labels).reshape(-1, 1)
        
    def __len__(self):
        return len(self.features) # 返回数据总量
    
    def __getitem__(self, idx):
        # 返回单条数据
        return self.features[idx], self.labels[idx]

# 创建仓库实例
train_ds = MyIncomeDataset(X_train, y_train)

# ==========================================
# 3. 定义 DataLoader (运输车：实现 Batch 的核心)
# ==========================================
# batch_size=64，意味着每一批取 64 条数据
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)

# DataLoader 随机生成 64 个索引，调用__getitem__() 从 Dataset 取出对应的数据，组成一批 (batch) 送到模型里。

# ==========================================
# 4. 定义模型
# ==========================================
model = nn.Sequential( #nn.Sequential 把多层网络串在一起，按顺序执行
    nn.Linear(len(numeric_cols), 32),
    nn.ReLU(),
    nn.Linear(32, 1),
    nn.Sigmoid()
).to(device)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

# ==========================================
# 5. 带有 Batch 的训练循环 (双层循环)
# ==========================================
epochs = 5  

print(f"总训练样本数: {len(train_ds)}")
print(f"每批次(Batch)大小: 64")
print(f"每轮(Epoch)需要的步数(Steps): {len(train_loader)}\n")
#总样本数除以 Batch 大小
# 每看 64 个样本，算出平均误差，更新一次参数。
# 1 步 (Step) = 1 个 Batch = 1 次参数更新


checkpoint_path = "latest_checkpoint.pth"
start_epoch = 0  # 默认从第 0 轮开始

# --- 【读档逻辑】 ---
if os.path.exists(checkpoint_path):
    print(f"检测到存档 {checkpoint_path}，正在恢复...")

    checkpoint = torch.load(checkpoint_path, map_location=device)
    
    # 1. 恢复脑子（权重）
    model.load_state_dict(checkpoint['model_state_dict'])
    # 2. 恢复肌肉记忆（优化器状态）
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    # 3. 恢复进度（轮数）
    start_epoch = checkpoint['epoch'] + 1 
    
    print(f"读档成功！从第 {start_epoch} 轮继续训练。")



# --- 【训练循环】 ---
for epoch in range(start_epoch, epochs):
    model.train()
    
    running_loss = 0.0 #?
    
    # 内层循环：DataLoader 自动帮你切分好了数据
    # batch_idx：当前是第几车
    # (data, target)：这一车里的 64 个特征和 64 个标签
    for batch_idx, (data, target) in enumerate(train_loader):
        
        data, target = data.to(device), target.to(device)
        # 标准五步法
        optimizer.zero_grad()
        output = model(data)           # 这里 data 的形状是 (64, 5)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() #.item()将包含一个元素的张量转换成一个标准的 Python 数值
        
        # 每隔 100 步打印一次，让你看到 Batch 的流动
        if (batch_idx + 1) % 100 == 0:
            print(f"Epoch {epoch+1} | Step {batch_idx+1}/{len(train_loader)} | Loss: {loss.item():.4f}")
            
    print(f"--- 第 {epoch+1} 轮结束，平均 Loss: {running_loss/len(train_loader):.4f} ---\n")
    
    # --- 【存档逻辑】 ---
    # 每轮结束，存一次档
    ckpt_data = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': loss.item(),
    }
    torch.save(ckpt_data, checkpoint_path)
    
    
torch.save(model.state_dict(), "model_weights.pth") # 保存模型权重
joblib.dump(scaler, 'scaler.joblib') # 推理阶段需要用它来标准化输入数据
print("训练完成！")

当前使用的设备: cpu
总训练样本数: 26048
每批次(Batch)大小: 64
每轮(Epoch)需要的步数(Steps): 407

检测到存档 latest_checkpoint.pth，正在恢复...
读档成功！从第 5 轮继续训练。
训练完成！


batch_size：通常设为 2 的幂次方（如 32, 64, 128, 256），这对 GPU 计算更友好。

shuffle=True：训练集必须开启。如果不打乱，模型可能会学到数据在表中的顺序（比如表中前 1000 条全是高收入），导致预测跑偏。测试集则通常设为 False。

drop_last：如果最后剩下的数据不够一个 Batch（比如剩 10 条，但 batch_size 是 64），是否丢掉它们。